In [1]:
"""
Phase 3: Training the Neural Cluster Router (Phi(q))
=====================================================
AD-BoN routing framework. Builds on:
  - Phase 1: dataset_clustered.csv, kmeans_model.joblib
  - Phase 2: junior_profile.json, senior_profile.json (not used directly here,
             but Phi(q)'s cluster outputs feed into the Phase 4 gating engine)

Trains a lightweight transformer sequence classifier to predict a soft
probability distribution over the K semantic clusters from Phase 1,
using temperature-scaled softmax targets derived from embedding-to-
centroid similarity (TO-Router style soft labels, not hard K-Means labels).

OUTPUT: ./saved_router/  (fine-tuned model + tokenizer + config)
        training_log.csv  (epoch-by-epoch train/val loss + val accuracy)

HOW TO USE
----------
1. Edit CONFIG below.
2. pip install torch transformers scikit-learn sentence-transformers pandas numpy joblib
3. python phase3_train_router.py
4. Use the ADBoNRouter class at the bottom (or import it) for inference.
"""

from __future__ import annotations

import logging
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("phase3")

# ============================== CONFIG ==============================
INPUT_PATH = "dataset_clustered.csv"
KMEANS_MODEL_PATH = "kmeans_model.joblib"

PROMPT_COL = "Prompt"
CLUSTER_COL = "cluster_id"

EMBEDDING_MODEL_NAME = "all-MiniLM-L12-v2"   # must match the model used in Phase 1
BASE_MODEL_NAME = "distilbert-base-uncased"  # lighter than deberta-v3-small; safer given ~930 train rows

SOFT_LABEL_TEMPERATURE = 2.0   # T in the temperature-scaled softmax
SIMILARITY_METRIC = "cosine"   # "cosine" or "euclidean"

TRAIN_FRAC = 0.8
VAL_FRAC = 0.1
TEST_FRAC = 0.1
RANDOM_STATE = 42

MAX_SEQ_LEN = 256
BATCH_SIZE = 16
NUM_EPOCHS = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 3   # stop if val loss doesn't improve for this many epochs

OUTPUT_DIR = "./saved_router"
TRAINING_LOG_PATH = "training_log.csv"
# ======================================================================

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


# ----------------------------------------------------------------------
# Step 1: Load assets
# ----------------------------------------------------------------------
def load_assets(csv_path: str, kmeans_path: str):
    import joblib

    p = Path(csv_path)
    if not p.exists():
        log.error(f"Input file not found: {p.resolve()}")
        sys.exit(1)

    df = pd.read_csv(p)
    required = {PROMPT_COL, CLUSTER_COL}
    missing = required - set(df.columns)
    if missing:
        log.error(f"Missing required column(s): {missing}. Found: {df.columns.tolist()}")
        sys.exit(1)

    km_p = Path(kmeans_path)
    if not km_p.exists():
        log.error(f"K-Means model not found: {km_p.resolve()}")
        sys.exit(1)

    kmeans = joblib.load(km_p)
    k = kmeans.cluster_centers_.shape[0]
    log.info(f"Loaded {len(df)} rows and K-Means model with K={k} clusters.")
    return df, kmeans, k


# ----------------------------------------------------------------------
# Step 2: Temperature-scaled soft label generation
# ----------------------------------------------------------------------
def compute_similarity_scores(embeddings: np.ndarray, centroids: np.ndarray, metric: str) -> np.ndarray:
    """Returns an (N, K) matrix of similarity scores between each embedding and each centroid."""
    if metric == "cosine":
        emb_norm = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-10)
        cen_norm = centroids / (np.linalg.norm(centroids, axis=1, keepdims=True) + 1e-10)
        return emb_norm @ cen_norm.T  # cosine similarity, higher = closer
    elif metric == "euclidean":
        # negative distance so higher = closer, consistent with cosine's convention
        dists = np.linalg.norm(embeddings[:, None, :] - centroids[None, :, :], axis=2)
        return -dists
    else:
        raise ValueError(f"Unknown SIMILARITY_METRIC: {metric}")


def temperature_softmax(scores: np.ndarray, temperature: float) -> np.ndarray:
    """
    y_i = exp(s_i / T) / sum_j exp(s_j / T), applied row-wise.
    Numerically stable via max-subtraction.
    """
    scaled = scores / temperature
    scaled -= scaled.max(axis=1, keepdims=True)
    exp_scores = np.exp(scaled)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)


def generate_soft_labels(prompts: List[str], centroids: np.ndarray) -> np.ndarray:
    """Embeds all prompts and converts centroid similarity into soft label distributions."""
    from sentence_transformers import SentenceTransformer

    device = "cuda" if torch.cuda.is_available() else "cpu"
    log.info(f"Embedding {len(prompts)} prompts with '{EMBEDDING_MODEL_NAME}' on {device} for soft-label generation...")
    embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)
    embeddings = embedder.encode(prompts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)

    scores = compute_similarity_scores(embeddings, centroids, SIMILARITY_METRIC)
    soft_labels = temperature_softmax(scores, SOFT_LABEL_TEMPERATURE)
    log.info(f"Generated soft labels with shape {soft_labels.shape} (T={SOFT_LABEL_TEMPERATURE}).")
    return soft_labels


# ----------------------------------------------------------------------
# Step 3: Dataset + splits
# ----------------------------------------------------------------------
class RouterDataset(Dataset):
    """Yields tokenized prompt inputs and a soft target distribution y in R^K."""

    def __init__(self, texts: List[str], soft_labels: np.ndarray, tokenizer, max_len: int):
        self.texts = texts
        self.soft_labels = soft_labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.soft_labels[idx], dtype=torch.float32)
        return item


def split_dataset(
    df: pd.DataFrame, soft_labels: np.ndarray
) -> Tuple[Tuple[List[str], np.ndarray], Tuple[List[str], np.ndarray], Tuple[List[str], np.ndarray]]:
    """
    80/10/10 train/val/test split, stratified on the hard cluster_id column
    (soft labels are continuous, so we stratify on the discrete K-Means
    assignment as the best available proxy for balanced splits).
    """
    from sklearn.model_selection import train_test_split

    indices = np.arange(len(df))
    strat_labels = df[CLUSTER_COL].values

    train_idx, temp_idx = train_test_split(
        indices, test_size=(1 - TRAIN_FRAC), random_state=RANDOM_STATE, stratify=strat_labels
    )
    val_size_of_temp = VAL_FRAC / (VAL_FRAC + TEST_FRAC)
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=(1 - val_size_of_temp),
        random_state=RANDOM_STATE,
        stratify=strat_labels[temp_idx],
    )

    texts = df[PROMPT_COL].tolist()

    def subset(idx):
        return [texts[i] for i in idx], soft_labels[idx]

    log.info(f"Split sizes -> train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_idx)}")
    return subset(train_idx), subset(val_idx), subset(test_idx)


# ----------------------------------------------------------------------
# Step 4: Model + custom KL-divergence Trainer
# ----------------------------------------------------------------------
class SoftLabelTrainer:
    """
    Lightweight custom training loop (AdamW + linear scheduler + KLDivLoss)
    rather than the full HF Trainer, so soft-label KL loss and early
    stopping are fully transparent and easy to modify.
    """

    def __init__(self, model, device: str):
        self.model = model.to(device)
        self.device = device
        self.kl_loss_fn = nn.KLDivLoss(reduction="batchmean")

    def compute_loss(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        labels = batch.pop("labels").to(self.device)
        inputs = {k: v.to(self.device) for k, v in batch.items()}

        outputs = self.model(**inputs)
        log_probs = torch.log_softmax(outputs.logits, dim=-1)  # KLDivLoss expects log-probs as input
        loss = self.kl_loss_fn(log_probs, labels)  # target is already a probability distribution
        return loss, outputs.logits

    @torch.no_grad()
    def evaluate(self, dataloader, hard_labels: np.ndarray) -> Tuple[float, float]:
        self.model.eval()
        total_loss, n_batches = 0.0, 0
        all_preds = []

        for batch in dataloader:
            loss, logits = self.compute_loss(dict(batch))
            total_loss += loss.item()
            n_batches += 1
            all_preds.append(torch.argmax(logits, dim=-1).cpu().numpy())

        preds = np.concatenate(all_preds)
        accuracy = float((preds == hard_labels).mean())
        avg_loss = total_loss / max(1, n_batches)
        return avg_loss, accuracy


def train_router(
    train_data, val_data, val_hard_labels: np.ndarray, k: int
) -> Tuple[nn.Module, "AutoTokenizer", pd.DataFrame]:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    from torch.utils.data import DataLoader
    from torch.optim import AdamW

    device = "cuda" if torch.cuda.is_available() else "cpu"
    log.info(f"Training on device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_NAME, num_labels=k)

    train_texts, train_labels = train_data
    val_texts, val_labels = val_data

    train_ds = RouterDataset(train_texts, train_labels, tokenizer, MAX_SEQ_LEN)
    val_ds = RouterDataset(val_texts, val_labels, tokenizer, MAX_SEQ_LEN)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * NUM_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(WARMUP_RATIO * total_steps), num_training_steps=total_steps
    )

    trainer = SoftLabelTrainer(model, device)

    best_val_loss = float("inf")
    epochs_without_improvement = 0
    log_rows = []

    for epoch in range(1, NUM_EPOCHS + 1):
        trainer.model.train()
        running_loss, n_batches = 0.0, 0

        for batch in train_loader:
            optimizer.zero_grad()
            loss, _ = trainer.compute_loss(dict(batch))
            loss.backward()
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()
            n_batches += 1

        train_loss = running_loss / max(1, n_batches)
        val_loss, val_acc = trainer.evaluate(val_loader, val_hard_labels)

        log.info(
            f"Epoch {epoch}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | val_accuracy={val_acc:.4f}"
        )
        log_rows.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_accuracy": val_acc})

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                log.info(f"Early stopping at epoch {epoch} (no val_loss improvement for {EARLY_STOPPING_PATIENCE} epochs).")
                break

    return trainer.model, tokenizer, pd.DataFrame(log_rows)


# ----------------------------------------------------------------------
# Step 6: Save model
# ----------------------------------------------------------------------
def save_router(model, tokenizer, output_dir: str) -> None:
    out = Path(output_dir)
    try:
        out.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(out)
        tokenizer.save_pretrained(out)
        log.info(f"Saved fine-tuned router to {out.resolve()}")
    except OSError as e:
        log.error(f"Failed to save router to {out}: {e}")
        raise


# ----------------------------------------------------------------------
# Step 7: Production runtime inference class
# ----------------------------------------------------------------------
class ADBoNRouter:
    """
    Production inference wrapper for the trained cluster router.
    Loads once, then call .route(query) to get Phi(q) in R^K.
    """

    def __init__(self, model_dir: str = OUTPUT_DIR, max_len: int = MAX_SEQ_LEN):
        from transformers import AutoTokenizer, AutoModelForSequenceClassification

        model_path = Path(model_dir)
        if not model_path.exists():
            raise FileNotFoundError(f"Router model directory not found: {model_path.resolve()}")

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_path).to(self.device)
        self.model.eval()
        self.max_len = max_len
        log.info(f"ADBoNRouter loaded from {model_path} on device={self.device}")

    @torch.no_grad()
    def route(self, query: str) -> np.ndarray:
        """Returns Phi(q): a normalized probability distribution over K clusters."""
        enc = self.tokenizer(
            query, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt"
        ).to(self.device)

        logits = self.model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        return probs.squeeze(0).cpu().numpy()

    @torch.no_grad()
    def route_batch(self, queries: List[str]) -> np.ndarray:
        """Batched version of route() for multiple queries at once."""
        enc = self.tokenizer(
            queries, truncation=True, padding=True, max_length=self.max_len, return_tensors="pt"
        ).to(self.device)
        logits = self.model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        return probs.cpu().numpy()


# ----------------------------------------------------------------------
# Orchestration
# ----------------------------------------------------------------------
def main() -> None:
    df, kmeans, k = load_assets(INPUT_PATH, KMEANS_MODEL_PATH)

    soft_labels = generate_soft_labels(df[PROMPT_COL].tolist(), kmeans.cluster_centers_)

    train_data, val_data, test_data = split_dataset(df, soft_labels)

    # Recover hard cluster_id labels for the val split (needed for accuracy metric)
    # by re-deriving indices; simpler here to just re-split cluster_id the same way.
    from sklearn.model_selection import train_test_split
    indices = np.arange(len(df))
    strat_labels = df[CLUSTER_COL].values
    train_idx, temp_idx = train_test_split(
        indices, test_size=(1 - TRAIN_FRAC), random_state=RANDOM_STATE, stratify=strat_labels
    )
    val_size_of_temp = VAL_FRAC / (VAL_FRAC + TEST_FRAC)
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=(1 - val_size_of_temp), random_state=RANDOM_STATE, stratify=strat_labels[temp_idx]
    )
    val_hard_labels = strat_labels[val_idx]
    test_hard_labels = strat_labels[test_idx]

    model, tokenizer, train_log_df = train_router(train_data, val_data, val_hard_labels, k)

    try:
        train_log_df.to_csv(TRAINING_LOG_PATH, index=False)
        log.info(f"Saved training log to {TRAINING_LOG_PATH}")
    except OSError as e:
        log.error(f"Failed to write training log: {e}")

    save_router(model, tokenizer, OUTPUT_DIR)

    # Quick test-set sanity check using the freshly saved router
    router = ADBoNRouter(OUTPUT_DIR)
    test_texts, _ = test_data
    test_probs = router.route_batch(test_texts)
    test_preds = np.argmax(test_probs, axis=1)
    test_acc = float((test_preds == test_hard_labels).mean())
    log.info(f"Test set accuracy (argmax vs hard K-Means cluster_id): {test_acc:.4f}")

    log.info("Phase 3 complete.")


if __name__ == "__main__":
    main()

2026-09-05 03:10:02,274 [INFO] Loaded 1162 rows and K-Means model with K=10 clusters.
2026-09-05 03:10:12,207 [INFO] Embedding 1162 prompts with 'all-MiniLM-L12-v2' on cpu for soft-label generation...
2026-09-05 03:10:13,275 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-09-05 03:10:13,278 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-05 03:10:13,725 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L12-v2/a50ef00143b4d5391434df20ae11632588ac25be/modules.json "HTTP/1.1 200 OK"
2026-09-05 03:10:13,981 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-09-05 03:10:14,142 [INFO] HTTP Request: HEAD 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-09-05 03:10:17,380 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-09-05 03:10:17,629 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-09-05 03:10:17,876 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-09-05 03:10:18,126 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2/resolve/main/preprocessor_config.json 

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

2026-09-05 03:10:28,838 [INFO] Generated soft labels with shape (1162, 10) (T=2.0).
2026-09-05 03:10:28,843 [INFO] Split sizes -> train: 929, val: 116, test: 117
2026-09-05 03:10:28,848 [INFO] Training on device: cpu
2026-09-05 03:10:29,909 [INFO] HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
2026-09-05 03:10:30,171 [INFO] HTTP Request: GET https://huggingface.co/distilbert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"


...

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-09-05 04:03:31,237 [INFO] Epoch 1/8 | train_loss=0.0041 | val_loss=0.0007 | val_accuracy=0.8362
2026-09-05 04:18:05,576 [INFO] Epoch 2/8 | train_loss=0.0009 | val_loss=0.0004 | val_accuracy=0.8707
2026-09-05 04:28:25,526 [INFO] E

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-09-05 05:15:13,182 [INFO] Saved fine-tuned router to C:\Users\abdul\OneDrive - SRM University, AP - Amaravathi\genesis\saved_router


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

2026-09-05 05:15:13,622 [INFO] ADBoNRouter loaded from saved_router on device=cpu
2026-09-05 05:15:15,610 [INFO] Test set accuracy (argmax vs hard K-Means cluster_id): 0.9316
2026-09-05 05:15:15,613 [INFO] Phase 3 complete.
